# Patient-level attention VAE

20 latent variables, about 10.4 million parameters, spatial and temporal attention. Demographics join the latent mean **only at the MI classifier**. The decoder reconstructs sampled XYZ, using latent geometry alone.

Frame repair and sampling are patient-local. Scaling, model fitting and early stopping stay inside each fold; final test patients are excluded. See the README for assumptions and configuration.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from mcpp.vae import CardiacVAE, VAEConfig
from mcpp.vae_data import DEMOGRAPHIC_COLUMNS, PreprocessConfig, load_cohort
from mcpp.vae_training import TrainConfig, cross_validate_vae, select_device

ROOT = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
PREPROCESS = PreprocessConfig(n_points=384)
MODEL = VAEConfig(n_points=PREPROCESS.n_points, latent_dim=20)
TRAINING = TrainConfig(epochs=60, batch_size=8)
N_FOLDS = 5
LIMIT_PATIENTS = None  # e.g. 40 for a two-fold smoke run
OUTPUT = ROOT / 'results/vae_training'
print('PyTorch:', torch.__version__, 'device:', select_device())
print('Parameters:', f'{CardiacVAE(MODEL).parameter_count():,}')
print('Head-only demographics:', DEMOGRAPHIC_COLUMNS)


## Prepare once

Retain 128 points in each anatomical region across all ten frames. Repair expansion and point scatter before sampling. Spatial coherence detects multi-frame defects; cycles with more than half their frames corrupted are excluded and recorded in the quality audit. This opens only the development patients' point arrays, one patient at a time. The existing 20% held-out test membership is preserved.


In [ ]:
metadata = pd.read_csv(ROOT / 'data/training_data/mixed_demographics.csv')
development, held_out = train_test_split(
    np.arange(len(metadata)), test_size=.2, stratify=metadata.MI, random_state=42,
)
if LIMIT_PATIENTS is not None and LIMIT_PATIENTS < len(development):
    development, _ = train_test_split(
        development, train_size=LIMIT_PATIENTS,
        stratify=metadata.MI.iloc[development], random_state=TRAINING.seed,
    )
xyz, regions, demographics, labels, repair_audit = load_cohort(
    ROOT / 'data/training_data', development, PREPROCESS,
)
development = np.asarray([a['patient_index'] for a in repair_audit if not a['excluded']], dtype=int)
quality_excluded = [a['patient_index'] for a in repair_audit if a['excluded']]
flagged = pd.DataFrame([
    {'patient': a['patient_index'], 'frames': a['flagged_frames'], 'excluded': a['excluded']}
    for a in repair_audit if a['flagged_frames']
], columns=['patient', 'frames', 'excluded'])
print('XYZ:', xyz.shape, 'demographics:', demographics.shape)
print('Test patients excluded:', len(held_out), 'patients with suspect frames:', len(flagged), 'excluded cycles:', len(quality_excluded))
flagged


## Train and cross-validate

This cell starts training. Every fold trains a fresh model and fits both scalers on its inner fitting patients. Inner validation selects the stopping epoch; the outer fold is scored once. Change `TRAINING`, `MODEL` or `N_FOLDS` above to experiment, and choose a fresh output directory. A two-epoch run checks the pipeline and does not establish predictive performance.


In [ ]:
if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError('Choose a new OUTPUT directory to preserve the prior experiment.')
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'frame_quality.json').write_text(json.dumps(repair_audit, indent=2))
results = cross_validate_vae(
    xyz, regions, demographics, labels, n_splits=N_FOLDS,
    model_config=MODEL, train_config=TRAINING, preprocess_config=PREPROCESS,
    output_dir=OUTPUT, patient_ids=development,
)
np.savez_compressed(
    OUTPUT / 'oof_predictions.npz', patient_ids=development, labels=labels,
    probabilities=results['oof_probabilities'], latent_means=results['oof_latent_means'],
)
summary = {key: value for key, value in results.items()
           if key not in ('oof_probabilities', 'oof_latent_means')}
summary.update(model_config=asdict(MODEL), train_config=asdict(TRAINING),
               preprocess_config=asdict(PREPROCESS), test_set_scored=False,
               test_patient_ids_excluded=held_out.tolist(),
               quality_excluded_patient_ids=quality_excluded,
               demographic_columns=list(DEMOGRAPHIC_COLUMNS),
               classification_threshold=.5, latent_axes_aligned_across_folds=False)
(OUTPUT / 'cv_results.json').write_text(json.dumps(summary, indent=2))
pd.DataFrame([{'fold': f['fold'], 'best_epoch': f['best_epoch'], **f['metrics']}
              for f in results['folds']])


In [ ]:
pd.DataFrame({
    'mean across folds': results['mean_fold_metrics'],
    'fold standard deviation': results['std_fold_metrics'],
    'pooled out-of-fold': results['oof_metrics'],
})


Each epoch prints training and inner-validation accuracy, pMI F1 and macro F1 at cutoff 0.5, alongside the losses and elapsed time. The scores are saved in each fold's `history` as `train_accuracy`, `train_f1_pMI`, `train_macro_f1`, `validation_accuracy`, `validation_f1_pMI` and `validation_macro_f1`. Training scores pool predictions from the training pass; validation scores use the epoch-end model with dropout disabled. Check reconstruction, KL and classification losses for collapse or overfitting. The saved checkpoints contain the demographic order, class mapping, geometry preprocessing, and train-fitted scaling. `predict_patients` in `mcpp.vae_training` accepts raw patient cycles and head-only demographics. Independent folds learn different latent coordinate systems; their latent axes are not aligned.
